# 🎯 4. Routing & Query Construction

Σε ένα μεγάλο RAG σύστημα έχουμε συχνά:

* **Πολλαπλές πηγές γνώσης** (πχ technical docs, FAQs, billing data)
* **Πολλαπλούς τύπους ερωτήσεων** (πχ συγκεκριμένα facts vs. how-to)
* **Δομημένα metadata** που θέλουμε να χρησιμοποιήσουμε ως filters

Σε αυτό το notebook μαθαίνουμε:

1. **Logical routing** : κατευθύνουμε queries σε διαφορετικές πηγές με structured output
2. **Semantic routing** : επιλογή prompt/strategy βάσει embedding similarity
3. **Query construction** : μετατροπή φυσικής γλώσσας σε δομημένο query με metadata filters

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(__file__).parent / ".env" if "__file__" in dir() else Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
# if not os.environ.get("OPENAI_API_KEY"):
#     import getpass
#     os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

LLM_MODEL   = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

## 4.1 Γιατί χρειαζόμαστε routing

Ας υποθέσουμε ότι η Datanous έχει **3 ξεχωριστά indices**:

* `technical_docs` — προδιαγραφές, APIs, configuration
* `pricing_docs` — χρεώσεις, plans, discounts
* `support_docs` — troubleshooting, FAQs

Αν τα ενώσουμε σε ένα τεράστιο index, το vocabulary μπερδεύεται και το embedding similarity γίνεται θόρυβος.

**Λύση:** Ένα **router** που, βάσει της ερώτησης, αποφασίζει σε **ποιο** index θα ψάξει.

<img src="images/routing_architecture.png" height="700px" width="900px" style="border-radius:10px;margin:12px 0;"/>

***Εικ. 4.1** — Multi-source routing: ο router αποφασίζει ποιο index να ρωτήσει. LLM-based (structured output) ή Embedding-based (cosine similarity).*

## 4.2 Logical Routing με structured output

Χρησιμοποιούμε function-calling του LLM για να επιστρέφει **τύπο** της ερώτησης ως ένα Pydantic enum.

In [3]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

class RouteDecision(BaseModel):
    """Route a user question to the correct Datanous.ai knowledge source."""
    datasource: Literal["product_docs", "pricing_docs", "support_docs"] = Field(
        ...,
        description=(
            "The knowledge source to use:\n"
            "- 'product_docs'  for features, APIs, integrations, and architecture\n"
            "- 'pricing_docs'  for plans, pricing tiers, discounts, and billing\n"
            "- 'support_docs'  for troubleshooting, errors, and operational issues"
        ),
    )
    reasoning: str = Field(description="Brief explanation of why this source was chosen")

llm        = ChatOpenAI(model=LLM_MODEL, temperature=0)
router_llm = llm.with_structured_output(RouteDecision)

ROUTER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert router for the Datanous.ai support system. "
     "Given a user question, select the most appropriate knowledge source: "
     "product_docs, pricing_docs, or support_docs."),
    ("human", "{question}"),
])

router = ROUTER_PROMPT | router_llm

# Test the router
test_questions = [
    "How does Datanous Guard detect prompt injection?",
    "What is the price of Datanous Insight Professional?",
    "Why is my Datanous Search API returning empty results?",
    "Does Datanous Embed support batch requests?",
]
for q in test_questions:
    decision = router.invoke({"question": q})
    print(f"Q: {q}")
    print(f"   -> {decision.datasource}  | {decision.reasoning[:70]}")


Q: How does Datanous Guard detect prompt injection?
   -> product_docs  | The question pertains to the features and capabilities of Datanous Gua
Q: What is the price of Datanous Insight Professional?
   -> pricing_docs  | The question is about the pricing of a specific product, which falls u
Q: Why is my Datanous Search API returning empty results?
   -> support_docs  | The question pertains to troubleshooting an issue with the Datanous Se
Q: Does Datanous Embed support batch requests?
   -> product_docs  | The question pertains to the features and capabilities of Datanous Emb


### Σύνδεση router με πραγματικές αλυσίδες

In [4]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

class RouteDecision(BaseModel):
    """Route a user question to the correct Datanous.ai knowledge source."""
    datasource: Literal["product_docs", "pricing_docs", "support_docs"] = Field(
        ...,
        description=(
            "The knowledge source to use:\n"
            "- 'product_docs'  for features, APIs, integrations, and architecture\n"
            "- 'pricing_docs'  for plans, pricing tiers, discounts, and billing\n"
            "- 'support_docs'  for troubleshooting, errors, and operational issues"
        ),
    )
    reasoning: str = Field(description="Brief explanation of why this source was chosen")

llm        = ChatOpenAI(model=LLM_MODEL, temperature=0)
router_llm = llm.with_structured_output(RouteDecision)

ROUTER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert router for the Datanous.ai support system. "
     "Given a user question, select the most appropriate knowledge source: "
     "product_docs, pricing_docs, or support_docs."),
    ("human", "{question}"),
])

router = ROUTER_PROMPT | router_llm

# Test the router
test_questions = [
    "How does Datanous Guard detect prompt injection?",
    "What is the price of Datanous Insight Professional?",
    "Why is my Datanous Search API returning empty results?",
    "Does Datanous Embed support batch requests?",
]
for q in test_questions:
    decision = router.invoke({"question": q})
    print(f"Q: {q}")
    print(f"   -> {decision.datasource}  | {decision.reasoning[:70]}")


Q: How does Datanous Guard detect prompt injection?
   -> product_docs  | The question pertains to the features and functionalities of Datanous 
Q: What is the price of Datanous Insight Professional?
   -> pricing_docs  | The question is about the pricing of a specific product, which falls u
Q: Why is my Datanous Search API returning empty results?
   -> support_docs  | The question pertains to troubleshooting an issue with the Datanous Se
Q: Does Datanous Embed support batch requests?
   -> product_docs  | The question pertains to the features and capabilities of Datanous Emb


## 4.3 Semantic Routing

Άλλος τρόπος επιλογής: αντί να ρωτάμε ένα LLM, μετράμε **embedding similarity** ανάμεσα στην ερώτηση και σε **prompt-templates**. Πιο γρήγορο και φθηνό.

Συχνή χρήση: **επιλογή προσωπικότητας/style** του απαντητή.

In [5]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

embedder = OpenAIEmbeddings(model=EMBED_MODEL)

# Three focused Chroma stores — one per knowledge domain
product_store = Chroma.from_documents(
    [
        Document(page_content="Datanous Insight ingests PDF, Word, Markdown, HTML, and plain-text formats via a hybrid indexing pipeline."),
        Document(page_content="Datanous Guard is a LangChain-compatible RunnableLambda performing grounding, injection detection, PII redaction, and tenant isolation."),
        Document(page_content="Datanous Embed supports text-embedding-3-small (1536 dims) and text-embedding-3-large (3072 dims) with Redis caching and batch size up to 2048."),
        Document(page_content="Datanous Search exposes a REST API with filter fields: document_type, department, date_range, and language. Returns ranked results in under 200 ms."),
    ],
    embedding=embedder, collection_name="product",
)

pricing_store = Chroma.from_documents(
    [
        Document(page_content="Datanous Insight Starter: 50 USD per month, up to 10,000 document pages, single-tenant, community support."),
        Document(page_content="Datanous Insight Professional: 350 USD per month, up to 100,000 pages, multi-tenant row-level access control, email support."),
        Document(page_content="Datanous Insight Enterprise: unlimited pages, dedicated vector store, 99.9 percent uptime SLA, custom pricing, on-premises option."),
        Document(page_content="All Datanous.ai plans include TLS 1.3 encryption in transit and AES-256 at rest. GDPR Data Processing Agreement included from Professional tier."),
    ],
    embedding=embedder, collection_name="pricing",
)

support_store = Chroma.from_documents(
    [
        Document(page_content="Empty results from Datanous Search: verify that the corpus has been re-indexed after document updates and that the metadata filter syntax matches the schema."),
        Document(page_content="Datanous Embed returning 429 errors: the batch size exceeds 2048. Split the input list into smaller batches and retry with exponential backoff."),
        Document(page_content="Datanous Guard blocking legitimate queries: review the grounding threshold. The default is 0.7; lowering it to 0.5 reduces false positives."),
        Document(page_content="Datanous Insight Professional showing cross-tenant data: ensure tenant_id is set correctly in the request header and that Guard middleware is enabled."),
    ],
    embedding=embedder, collection_name="support",
)

store_map = {
    "product_docs":  product_store.as_retriever(search_kwargs={"k": 2}),
    "pricing_docs":  pricing_store.as_retriever(search_kwargs={"k": 2}),
    "support_docs":  support_store.as_retriever(search_kwargs={"k": 2}),
}

RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are a technical assistant for Datanous.ai.
Answer using ONLY the context below. If not found, say "I do not have that information."

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def routed_rag(question: str) -> str:
    """Route question -> select store -> retrieve -> generate."""
    decision  = router.invoke({"question": question})
    retriever = store_map[decision.datasource]
    docs      = retriever.invoke(question)
    context   = format_docs(docs)
    return (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )

# Test routed RAG
for q in [
    "What document formats does Datanous Insight support?",
    "How much does the Professional plan cost?",
    "Why is Datanous Guard blocking my queries with a 0.7 threshold?",
]:
    print(f"Q: {q}")
    print(f"A: {routed_rag(q)}\n")


Q: What document formats does Datanous Insight support?
A: Datanous Insight supports PDF, Word, Markdown, HTML, and plain-text formats.

Q: How much does the Professional plan cost?
A: The Professional plan costs 350 USD per month.

Q: Why is Datanous Guard blocking my queries with a 0.7 threshold?
A: Datanous Guard is blocking your queries because the grounding threshold is set to 0.7, which may be resulting in false positives. Lowering the threshold to 0.5 could help reduce these false positives.



# Αλέξανδρος | Semantic Routing extra

### Semantic router based only on embeddings and cosine similarity, without a generative LLM call.

In [6]:
from langchain_openai import OpenAIEmbeddings
import numpy as np

# Αντιπροσωπευτικές "anchor" φράσεις για κάθε route
ROUTE_ANCHORS = {
    "product_docs":  "API features architecture integration documentation",
    "pricing_docs":  "price cost plan billing subscription tier",
    "support_docs":  "error bug timeout failed troubleshooting fix",
}

embedder = OpenAIEmbeddings(model=EMBED_MODEL)
anchor_embeddings = {
    name: embedder.embed_query(text)
    for name, text in ROUTE_ANCHORS.items()
}

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_route(question: str) -> tuple[str, dict]:
    q_emb  = embedder.embed_query(question)
    scores = {
        name: cosine_similarity(q_emb, anchor)
        for name, anchor in anchor_embeddings.items()
    }
    return max(scores, key=scores.get), scores


In [7]:
# Test: semantic router χωρίς LLM call 
test_questions = [
    "How does Datanous Guard detect prompt injection?",
    "What is the price of the Professional plan?",
    "Why is my Datanous Search API returning empty results?",
    "Does Datanous Embed support batch requests?",
]

print("=== Semantic Router (embedding-only, no LLM) ===\n")
for q in test_questions:
    chosen, scores = semantic_route(q)
    print(f"Q: {q}")
    for name, score in sorted(scores.items(), key=lambda x: -x[1]):
        marker = " ◄" if name == chosen else ""
        print(f"   {name:<15} {score:.4f}{marker}")
    print()


=== Semantic Router (embedding-only, no LLM) ===

Q: How does Datanous Guard detect prompt injection?
   support_docs    0.2544 ◄
   product_docs    0.1839
   pricing_docs    0.0820

Q: What is the price of the Professional plan?
   pricing_docs    0.5350 ◄
   product_docs    0.2087
   support_docs    0.0873

Q: Why is my Datanous Search API returning empty results?
   product_docs    0.2532 ◄
   support_docs    0.1995
   pricing_docs    0.0802

Q: Does Datanous Embed support batch requests?
   product_docs    0.2874 ◄
   pricing_docs    0.1976
   support_docs    0.1798



### Semantic Search using sklearn 

In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

route_names = list(anchor_embeddings.keys())
anchor_matrix = np.array([anchor_embeddings[name] for name in route_names])

def semantic_route_sk(question: str):
    q_emb = np.array(embedder.embed_query(question)).reshape(1, -1)

    sims = cosine_similarity(q_emb, anchor_matrix)[0]

    scores = {
        name: float(score)
        for name, score in zip(route_names, sims)
    }

    chosen = max(scores, key=scores.get)
    return chosen, scores

In [9]:
# Test: semantic router χωρίς LLM call 
test_questions = [
    "How does Datanous Guard detect prompt injection?",
    "What is the price of the Professional plan?",
    "Why is my Datanous Search API returning empty results?",
    "Does Datanous Embed support batch requests?",
]

print("=== Semantic Router (embedding-only, no LLM) ===\n")
for q in test_questions:
    chosen, scores = semantic_route_sk(q)
    print(f"Q: {q}")
    for name, score in sorted(scores.items(), key=lambda x: -x[1]):
        marker = " ◄" if name == chosen else ""
        print(f"   {name:<15} {score:.4f}{marker}")
    print()

=== Semantic Router (embedding-only, no LLM) ===

Q: How does Datanous Guard detect prompt injection?
   support_docs    0.2544 ◄
   product_docs    0.1839
   pricing_docs    0.0820

Q: What is the price of the Professional plan?
   pricing_docs    0.5350 ◄
   product_docs    0.2087
   support_docs    0.0873

Q: Why is my Datanous Search API returning empty results?
   product_docs    0.2532 ◄
   support_docs    0.1995
   pricing_docs    0.0802

Q: Does Datanous Embed support batch requests?
   product_docs    0.2874 ◄
   pricing_docs    0.1976
   support_docs    0.1798



### -> Check when Semantic Router can fail

In [10]:
import time
import numpy as np
from IPython.display import display, Markdown
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI


def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_route(question: str) -> tuple[str, dict]:
    q_emb  = embedder.embed_query(question)
    scores = {
        name: cosine_sim(q_emb, anchor)
        for name, anchor in anchor_embeddings.items()
    }
    return max(scores, key=scores.get), scores

def generate_answer(question: str, datasource: str) -> str:
    """Retrieve from the chosen store and generate an answer."""


    _llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

    _prompt = ChatPromptTemplate.from_template(
        """You are a technical assistant for Datanous.ai.
Answer using ONLY the context below. If not found, say "I do not have that information."

Context:
{context}

Question: {question}

Answer:"""
    )

    retriever = store_map[datasource]
    docs      = retriever.invoke(question)
    context   = "\n\n".join(d.page_content for d in docs)

    return (_prompt | _llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )


questions = [
    # "How does Datanous Guard detect prompt injection?",
    "What is the price of the Professional plan?",
    "Why is my Datanous Search API returning empty results?",
    "Is the Enterprise plan worth it for heavy API usage?",   # ambiguous
    "Getting 429 errors — is this a billing issue?",          # ambiguous
]

results = []

for q in questions:
    # Embedding router
    t0 = time.time()
    emb_choice, scores = semantic_route(q)
    emb_time = (time.time() - t0) * 1000
    emb_answer = generate_answer(q, emb_choice)

    # LLM router
    t1 = time.time()
    llm_decision = router.invoke({"question": q})
    llm_choice   = llm_decision.datasource
    llm_time     = (time.time() - t1) * 1000
    llm_answer   = generate_answer(q, llm_choice)

    match = "OK" if emb_choice == llm_choice else "ΔΙΑΦΟΡΑ"
    results.append((q, emb_choice, emb_time, scores, emb_answer,
                    llm_choice, llm_time, llm_decision.reasoning, llm_answer, match))

# --- Markdown Output ---
md = "## Router Comparison: Embedding vs LLM\n\n"

for q, emb, et, scores, emb_ans, llm, lt, reasoning, llm_ans, match in results:
    status = "**match**" if match == "OK" else "**ΔΙΑΦΟΡΑ**"
    md += f"---\n\n### Q: {q}\n\n"

    # Embedding router
    md += f"**Embedding Router** &rarr; `{emb}` &nbsp;|&nbsp; _{et:.0f} ms_\n\n"
    md += "| Source | Score | Bar |\n|---|---|---|\n"
    for name, sc in sorted(scores.items(), key=lambda x: -x[1]):
        bar = "#" * int(sc * 30)
        marker = " **<**" if name == emb else ""
        md += f"| `{name}` | {sc:.4f} | `{bar}`{marker} |\n"
    md += f"\n**Answer (Embedding):** {emb_ans}\n\n"

    # LLM router
    md += f"**LLM Router** &rarr; `{llm}` &nbsp;|&nbsp; _{lt:.0f} ms_\n\n"
    md += f"> Reasoning: {reasoning}\n\n"
    md += f"**Answer (LLM):** {llm_ans}\n\n"

    # Result
    md += f"**Result:** {status}"
    if match == "ΔΙΑΦΟΡΑ":
        md += " — Οι δύο routers επέλεξαν διαφορετικό index, οι απαντήσεις φαίνονται παραπάνω."
    md += "\n\n"

# Summary
avg_emb = sum(r[2] for r in results) / len(results)
avg_llm = sum(r[6] for r in results) / len(results)
matches = sum(1 for r in results if r[9] == "OK")

md += "---\n\n## Summary\n\n"
md += "| Metric | Value |\n|---|---|\n"
md += f"| Avg Embedding Router latency | {avg_emb:.0f} ms |\n"
md += f"| Avg LLM Router latency | {avg_llm:.0f} ms |\n"
md += f"| Speed advantage | {avg_llm/avg_emb:.1f}x faster (embedding) |\n"
md += f"| Agreement | {matches}/{len(results)} questions |\n\n"
md += "> **Note:** When the routers disagree, compare the two answers above. "
md += "The LLM router understands **intent**, the embedding router matches **vocabulary**."

display(Markdown(md))


## Router Comparison: Embedding vs LLM

---

### Q: What is the price of the Professional plan?

**Embedding Router** &rarr; `pricing_docs` &nbsp;|&nbsp; _193 ms_

| Source | Score | Bar |
|---|---|---|
| `pricing_docs` | 0.5350 | `################` **<** |
| `product_docs` | 0.2087 | `######` |
| `support_docs` | 0.0873 | `##` |

**Answer (Embedding):** The price of the Professional plan is 350 USD per month.

**LLM Router** &rarr; `pricing_docs` &nbsp;|&nbsp; _1520 ms_

> Reasoning: The question is about the pricing of a specific plan, which falls under the category of pricing information.

**Answer (LLM):** The price of the Professional plan is 350 USD per month.

**Result:** **match**

---

### Q: Why is my Datanous Search API returning empty results?

**Embedding Router** &rarr; `product_docs` &nbsp;|&nbsp; _145 ms_

| Source | Score | Bar |
|---|---|---|
| `product_docs` | 0.2532 | `#######` **<** |
| `support_docs` | 0.1995 | `#####` |
| `pricing_docs` | 0.0802 | `##` |

**Answer (Embedding):** I do not have that information.

**LLM Router** &rarr; `support_docs` &nbsp;|&nbsp; _1479 ms_

> Reasoning: The question pertains to troubleshooting an issue with the Datanous Search API, which falls under operational issues and errors. Therefore, support documentation is the most appropriate source.

**Answer (LLM):** Your Datanous Search API is returning empty results because you need to verify that the corpus has been re-indexed after document updates and that the metadata filter syntax matches the schema.

**Result:** **ΔΙΑΦΟΡΑ** — Οι δύο routers επέλεξαν διαφορετικό index, οι απαντήσεις φαίνονται παραπάνω.

---

### Q: Is the Enterprise plan worth it for heavy API usage?

**Embedding Router** &rarr; `product_docs` &nbsp;|&nbsp; _136 ms_

| Source | Score | Bar |
|---|---|---|
| `product_docs` | 0.4193 | `############` **<** |
| `pricing_docs` | 0.4139 | `############` |
| `support_docs` | 0.1607 | `####` |

**Answer (Embedding):** I do not have that information.

**LLM Router** &rarr; `pricing_docs` &nbsp;|&nbsp; _1213 ms_

> Reasoning: The question pertains to the value and benefits of the Enterprise plan, which relates to pricing tiers and what is included in that plan, especially in the context of heavy API usage.

**Answer (LLM):** I do not have that information.

**Result:** **ΔΙΑΦΟΡΑ** — Οι δύο routers επέλεξαν διαφορετικό index, οι απαντήσεις φαίνονται παραπάνω.

---

### Q: Getting 429 errors — is this a billing issue?

**Embedding Router** &rarr; `pricing_docs` &nbsp;|&nbsp; _145 ms_

| Source | Score | Bar |
|---|---|---|
| `pricing_docs` | 0.3196 | `#########` **<** |
| `support_docs` | 0.3171 | `#########` |
| `product_docs` | 0.2009 | `######` |

**Answer (Embedding):** I do not have that information.

**LLM Router** &rarr; `support_docs` &nbsp;|&nbsp; _1223 ms_

> Reasoning: The question pertains to receiving 429 errors, which typically indicate too many requests and are related to operational issues rather than billing or pricing.

**Answer (LLM):** I do not have that information.

**Result:** **ΔΙΑΦΟΡΑ** — Οι δύο routers επέλεξαν διαφορετικό index, οι απαντήσεις φαίνονται παραπάνω.

---

## Summary

| Metric | Value |
|---|---|
| Avg Embedding Router latency | 155 ms |
| Avg LLM Router latency | 1359 ms |
| Speed advantage | 8.8x faster (embedding) |
| Agreement | 1/4 questions |

> **Note:** When the routers disagree, compare the two answers above. The LLM router understands **intent**, the embedding router matches **vocabulary**.

<img src="images/routing_decision.png" width="50%" style="border-radius:10px;margin:12px 0;"/>


In [11]:
import re
import numpy as np
from dataclasses import dataclass
from IPython.display import display, Markdown


@dataclass
class RoutingResult:
    datasource: str
    method: str        # "rules", "embedding", "llm"
    confidence: float  # 1.0 για rules, cosine score για embedding, 1.0 για llm
    reasoning: str


class HybridRouter:
    """
    3-tier routing:
      Tier 1 — Keyword rules   : deterministic, zero API cost
      Tier 2 — Embedding sim   : fast + cheap, no LLM
      Tier 3 — LLM fallback    : accurate, used only when needed
    """

    # Tier 1: keyword rules
    # re.compile(r"\b(price|cost|usd|plan|billing|tier|discount|subscription|worth)\b", re.I)
    # ταιριάζει ερωτήσεις που περιέχουν μία από αυτές τις λέξεις: price, cost, usd ...
    # Το \b σημαίνει word boundary
    # r"api" μπορεί να κάνει match μέσα σε λέξη όπως: capillary
    # Το re.I σημαίνει ignore case.
    RULES = [
        (re.compile(r"\b(price|cost|usd|plan|billing|tier|discount|subscription|worth)\b", re.I),
         "pricing_docs",  "pricing keyword"),
        (re.compile(r"\b(error|exception|failed|timeout|429|503|404|not working|issue|bug|blocking)\b", re.I),
         "support_docs",  "support keyword"),
        (re.compile(r"\b(api|sdk|configure|architecture|integration|endpoint|detect|support|format)\b", re.I),
         "product_docs",  "product keyword"),
    ]

    # Tier 2: embedding confidence threshold
    EMBEDDING_THRESHOLD = 0.35

    def __init__(self, embedder, anchor_embeddings, llm_router, cosine_fn):
        self.embedder          = embedder
        self.anchor_embeddings = anchor_embeddings
        self.llm_router        = llm_router   # LangChain chain (router από 4.2)
        self.cosine_fn         = cosine_fn
        self.stats             = {"rules": 0, "embedding": 0, "llm": 0}

    def route(self, question: str) -> RoutingResult:

        # Tier 1: Keyword rules
        for pattern, datasource, label in self.RULES:
            if pattern.search(question):
                self.stats["rules"] += 1
                return RoutingResult(
                    datasource=datasource,
                    method="rules",
                    confidence=1.0,
                    reasoning=f"Matched rule: {label}",
                )

        # Tier 2: Embedding similarity
        q_emb  = self.embedder.embed_query(question)
        scores = {
            name: self.cosine_fn(q_emb, anchor)
            for name, anchor in self.anchor_embeddings.items()
        }
        best_name  = max(scores, key=scores.get)
        best_score = scores[best_name]

        if best_score >= self.EMBEDDING_THRESHOLD:
            self.stats["embedding"] += 1
            return RoutingResult(
                datasource=best_name,
                method="embedding",
                confidence=best_score,
                reasoning=f"Cosine similarity {best_score:.4f} >= threshold {self.EMBEDDING_THRESHOLD}",
            )

        # Tier 3: LLM fallback
        decision = self.llm_router.invoke({"question": question})
        self.stats["llm"] += 1
        return RoutingResult(
            datasource=decision.datasource,
            method="llm",
            confidence=1.0,
            reasoning=decision.reasoning,
        )

    def stats_report(self) -> str:
        total = sum(self.stats.values()) or 1
        lines = ["| Tier | Method | Count | % |", "|---|---|---|---|"]
        labels = {"rules": "Tier 1 — Keyword Rules", "embedding": "Tier 2 — Embedding", "llm": "Tier 3 — LLM"}
        for key, label in labels.items():
            n = self.stats[key]
            lines.append(f"| {label} | `{key}` | {n} | {n/total*100:.0f}% |")
        return "\n".join(lines)


# Instantiate 
hybrid = HybridRouter(
    embedder=embedder,
    anchor_embeddings=anchor_embeddings,
    llm_router=router,
    cosine_fn=cosine_sim,
)

# Test questions
test_questions = [
    # ── Tier 1 expected: ξεκάθαρα keywords 
    "What is the monthly fee for the Professional subscription?",  
    "My requests are failing with a 429 response",                 

    # Tier 2 expected: semantic αλλά χωρίς keyword
    "How do I ingest documents into Datanous Insight?",           
    "What output dimensions does Datanous Embed produce?",         

    # Tier 3 expected: αμφίσημες, χαμηλό cosine score
    "Is Datanous a good fit for a regulated financial institution?", 
    "We had a slow response yesterday, should we upgrade?",          
]


md = "## Hybrid Router: 3-Tier Decision\n\n"
md += "| Tier | Method | Description |\n|---|---|---|\n"
md += "| 1 | `rules` | Keyword regex — deterministic, zero API cost |\n"
md += "| 2 | `embedding` | Cosine similarity — fast, no LLM |\n"
md += "| 3 | `llm` | LLM fallback — accurate, used only when needed |\n\n"

for q in test_questions:
    result = hybrid.route(q)
    tier   = {"rules": "Tier 1", "embedding": "Tier 2", "llm": "Tier 3"}[result.method]
    md += f"---\n\n**Q:** {q}\n\n"
    md += f"- **Tier:** {tier} (`{result.method}`)\n"
    md += f"- **Datasource:** `{result.datasource}`\n"
    md += f"- **Confidence:** {result.confidence:.4f}\n"
    md += f"- **Reasoning:** {result.reasoning}\n\n"

md += "---\n\n## Routing Stats\n\n"
md += hybrid.stats_report()
md += "\n\n> Tier 1 handles the obvious cases for free. "
md += "Tier 2 covers the rest cheaply. "
md += "Tier 3 (LLM) fires only for genuinely ambiguous questions."

display(Markdown(md))


## Hybrid Router: 3-Tier Decision

| Tier | Method | Description |
|---|---|---|
| 1 | `rules` | Keyword regex — deterministic, zero API cost |
| 2 | `embedding` | Cosine similarity — fast, no LLM |
| 3 | `llm` | LLM fallback — accurate, used only when needed |

---

**Q:** What is the monthly fee for the Professional subscription?

- **Tier:** Tier 1 (`rules`)
- **Datasource:** `pricing_docs`
- **Confidence:** 1.0000
- **Reasoning:** Matched rule: pricing keyword

---

**Q:** My requests are failing with a 429 response

- **Tier:** Tier 1 (`rules`)
- **Datasource:** `support_docs`
- **Confidence:** 1.0000
- **Reasoning:** Matched rule: support keyword

---

**Q:** How do I ingest documents into Datanous Insight?

- **Tier:** Tier 2 (`embedding`)
- **Datasource:** `product_docs`
- **Confidence:** 0.3555
- **Reasoning:** Cosine similarity 0.3555 >= threshold 0.35

---

**Q:** What output dimensions does Datanous Embed produce?

- **Tier:** Tier 3 (`llm`)
- **Datasource:** `product_docs`
- **Confidence:** 1.0000
- **Reasoning:** The question pertains to the features and specifications of the Datanous Embed, which falls under product documentation.

---

**Q:** Is Datanous a good fit for a regulated financial institution?

- **Tier:** Tier 3 (`llm`)
- **Datasource:** `product_docs`
- **Confidence:** 1.0000
- **Reasoning:** The question pertains to the suitability of Datanous for a specific industry, which involves understanding the product's features, compliance capabilities, and integrations that may be relevant for regulated financial institutions.

---

**Q:** We had a slow response yesterday, should we upgrade?

- **Tier:** Tier 3 (`llm`)
- **Datasource:** `pricing_docs`
- **Confidence:** 1.0000
- **Reasoning:** The user is inquiring about upgrading, which relates to plans, pricing tiers, and potentially the benefits of upgrading to improve performance.

---

## Routing Stats

| Tier | Method | Count | % |
|---|---|---|---|
| Tier 1 — Keyword Rules | `rules` | 2 | 33% |
| Tier 2 — Embedding | `embedding` | 1 | 17% |
| Tier 3 — LLM | `llm` | 3 | 50% |

> Tier 1 handles the obvious cases for free. Tier 2 covers the rest cheaply. Tier 3 (LLM) fires only for genuinely ambiguous questions.

## 4.4 Query Construction — από φυσική γλώσσα σε δομημένα queries

Φαντάσου ότι έχεις μια vector store με metadata όπως:

```yaml
doc_id: "INC-2024-0431"
product: "orbit_storage"
severity: "critical"
region: "eu-west-1"
created_at: "2024-09-12"
resolved: true
duration_minutes: 47
```

Ένας χρήστης ρωτά: *«Δείξε critical incidents του Orbit Storage στην Ευρώπη μετά τον Μάιο 2024 που διήρκεσαν >30 λεπτά»*

Αυτή η ερώτηση δεν είναι semantic search — είναι **structured query**. Με embedding-only retrieval θα αποτυγχάναμε.

**Λύση:** Χρησιμοποιούμε LLM για να μετατρέψουμε την ερώτηση σε **structured filter** (Pydantic), και η vector store κάνει retrieval ΜΕ filter.

In [11]:
from datetime import date
from typing import Optional

llm_client = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class SupportTicketQuery(BaseModel):
    """Structured query for Datanous.ai support ticket database."""

    text_search: str = Field(...,description="Semantic search query for the ticket description",)
    product: Optional[Literal["Datanous Insight", "Datanous Search", "Datanous Guard", "Datanous Embed"]] = Field(None,description="Datanous.ai product name if explicitly mentioned",)
    severity: Optional[Literal["low", "medium", "high", "critical"]] = Field(None,description="Ticket severity if mentioned",)
    tenant_tier: Optional[Literal["Starter", "Professional", "Enterprise"]] = Field(None,description="Subscription tier of the affected tenant if mentioned",)
    created_after: Optional[date] = Field(None,description="Only tickets created after this date",)
    created_before: Optional[date] = Field(None,description="Only tickets created before this date",)
    resolved: Optional[bool] = Field(None,description="Filter by resolution status if mentioned",)
    min_duration_minutes: Optional[int] = Field(None,description="Only tickets whose duration is at least this many minutes",)
    max_duration_minutes: Optional[int] = Field(None,description="Only tickets whose duration is at most this many minutes",)

TICKET_QUERY_PROMPT = ChatPromptTemplate.from_template(
    """Convert the user request into a structured SupportTicketQuery for Datanous.ai.
Extract only what is explicitly mentioned.

User request: {question}"""
)

ticket_query_chain = (
    TICKET_QUERY_PROMPT
    | llm_client.with_structured_output(SupportTicketQuery)
)

# Test query construction
test_requests = [
    "Show me critical unresolved Datanous Guard issues from the last month",
    "Find high-severity Datanous Search tickets for Enterprise clients",
    "Any Datanous Embed errors reported after June 2024?",
    "Show tickets that lasted more than 30 minutes",
]

for req in test_requests:
    q = ticket_query_chain.invoke({"question": req})
    print(f"Request : {req}")
    print(f"  text_search : {q.text_search}")
    print(f"  product     : {q.product}")
    print(f"  severity    : {q.severity}")
    print(f"  tenant_tier : {q.tenant_tier}")
    print(f"  resolved    : {q.resolved}")
    print(f"  created_after : {q.created_after}")
    print(f"  min_duration : {q.min_duration_minutes}")
    print()

Request : Show me critical unresolved Datanous Guard issues from the last month
  text_search : 
  product     : Datanous Guard
  severity    : critical
  tenant_tier : None
  resolved    : False
  created_after : 2023-09-01
  min_duration : None

Request : Find high-severity Datanous Search tickets for Enterprise clients
  text_search : 
  product     : Datanous Search
  severity    : high
  tenant_tier : Enterprise
  resolved    : None
  created_after : None
  min_duration : None

Request : Any Datanous Embed errors reported after June 2024?
  text_search : errors
  product     : Datanous Embed
  severity    : None
  tenant_tier : None
  resolved    : None
  created_after : 2024-06-01
  min_duration : None

Request : Show tickets that lasted more than 30 minutes
  text_search : 
  product     : None
  severity    : None
  tenant_tier : None
  resolved    : None
  created_after : None
  min_duration : 30



### Εκτέλεση structured query σε vector store

Τα filters του Chroma δέχονται operators όπως `$gte`, `$lte`, `$eq`, `$in`, `$and`.

In [12]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

# Synthetic support-ticket corpus with metadata.
# This is the vector store used by both the manual structured-query demo
# and the SelfQueryRetriever demo in the next section.
ticket_docs = [
    Document(
        page_content="Critical unresolved incident: Datanous Guard blocked legitimate tenant requests after a policy update.",
        metadata={
            "ticket_id": "INC-2024-0431",
            "product": "Datanous Guard",
            "severity": "critical",
            "tenant_tier": "Enterprise",
            "created_at": "2024-09-12",
            "resolved": False,
            "duration_minutes": 47,
        },
    ),
    Document(
        page_content="High severity Datanous Search issue: metadata filter syntax caused empty result sets for a Professional tenant.",
        metadata={
            "ticket_id": "INC-2024-0442",
            "product": "Datanous Search",
            "severity": "high",
            "tenant_tier": "Professional",
            "created_at": "2024-10-03",
            "resolved": True,
            "duration_minutes": 18,
        },
    ),
    Document(
        page_content="Datanous Embed returned 429 errors because the client exceeded the maximum batch size.",
        metadata={
            "ticket_id": "INC-2024-0450",
            "product": "Datanous Embed",
            "severity": "medium",
            "tenant_tier": "Starter",
            "created_at": "2024-10-21",
            "resolved": True,
            "duration_minutes": 22,
        },
    ),
    Document(
        page_content="Datanous Insight Professional tenant reported cross-tenant rows because tenant_id middleware was disabled.",
        metadata={
            "ticket_id": "INC-2024-0463",
            "product": "Datanous Insight",
            "severity": "critical",
            "tenant_tier": "Professional",
            "created_at": "2024-11-05",
            "resolved": False,
            "duration_minutes": 63,
        },
    ),
    Document(
        page_content="Low severity Datanous Search timeout affected an Enterprise tenant during a regional index rebuild.",
        metadata={
            "ticket_id": "INC-2024-0471",
            "product": "Datanous Search",
            "severity": "low",
            "tenant_tier": "Enterprise",
            "created_at": "2024-11-18",
            "resolved": True,
            "duration_minutes": 9,
        },
    ),
]

tickets_store = Chroma.from_documents(
    ticket_docs,
    embedding=embedder,
    collection_name="datanous_support_tickets",
)


def build_chroma_filter(q: SupportTicketQuery) -> dict | None:
    """Translate our Pydantic query object into a Chroma metadata filter."""
    clauses = []

    if q.product:
        clauses.append({"product": {"$eq": q.product}})
    if q.severity:
        clauses.append({"severity": {"$eq": q.severity}})
    if q.tenant_tier:
        clauses.append({"tenant_tier": {"$eq": q.tenant_tier}})
    if q.resolved is not None:
        clauses.append({"resolved": {"$eq": q.resolved}})
    if q.created_after:
        clauses.append({"created_at": {"$gte": q.created_after.isoformat()}})
    if q.created_before:
        clauses.append({"created_at": {"$lte": q.created_before.isoformat()}})
    if q.min_duration_minutes is not None:
        clauses.append({"duration_minutes": {"$gte": q.min_duration_minutes}})
    if q.max_duration_minutes is not None:
        clauses.append({"duration_minutes": {"$lte": q.max_duration_minutes}})

    if not clauses:
        return None
    if len(clauses) == 1:
        return clauses[0]
    return {"$and": clauses}


def run_ticket_search(question: str, k: int = 5):
    """Natural language -> structured query -> Chroma similarity search + metadata filter."""
    structured_query = ticket_query_chain.invoke({"question": question})
    chroma_filter = build_chroma_filter(structured_query)

    print(f"Question: {question}")
    print(f"Text search: {structured_query.text_search}")
    print(f"Chroma filter: {chroma_filter}\n")

    results = tickets_store.similarity_search(
        structured_query.text_search,
        k=k,
        filter=chroma_filter,
    )

    if not results:
        print("No matching tickets found.\n")
        return []

    for doc in results:
        m = doc.metadata
        print(
            f"  {m['ticket_id']} | {m['product']} | {m['severity']} | "
            f"tier={m['tenant_tier']} | resolved={m['resolved']} | "
            f"duration={m['duration_minutes']} min"
        )
        print(f"  {doc.page_content}\n")

    return results


# Test manual structured retrieval
for question in [
    "unresolved critical Datanous Guard tickets",
    "high severity Datanous Search issues for Professional clients",
    "any Datanous Embed tickets that lasted more than 15 minutes",
]:
    run_ticket_search(question)

Question: unresolved critical Datanous Guard tickets
Text search: unresolved
Chroma filter: {'$and': [{'product': {'$eq': 'Datanous Guard'}}, {'severity': {'$eq': 'critical'}}, {'resolved': {'$eq': False}}]}

  INC-2024-0431 | Datanous Guard | critical | tier=Enterprise | resolved=False | duration=47 min
  Critical unresolved incident: Datanous Guard blocked legitimate tenant requests after a policy update.

Question: high severity Datanous Search issues for Professional clients
Text search: Datanous Search issues
Chroma filter: {'$and': [{'product': {'$eq': 'Datanous Search'}}, {'severity': {'$eq': 'high'}}, {'tenant_tier': {'$eq': 'Professional'}}]}

  INC-2024-0442 | Datanous Search | high | tier=Professional | resolved=True | duration=18 min
  High severity Datanous Search issue: metadata filter syntax caused empty result sets for a Professional tenant.

Question: any Datanous Embed tickets that lasted more than 15 minutes
Text search: 
Chroma filter: {'$and': [{'product': {'$eq': 

## 4.5 Self-Querying Retriever

Το LangChain παρέχει έτοιμο **`SelfQueryRetriever`** που κάνει αυτό αυτόματα: ορίζεις το **schema** και αυτός φτιάχνει το structured query.

In [13]:
# Required once for SelfQueryRetriever's structured-query parser:
# %pip install -U lark langchain-classic

from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo

metadata_field_info = [
    AttributeInfo(
        name="product",
        description="Datanous.ai product name. One of: Datanous Insight, Datanous Search, Datanous Guard, Datanous Embed.",
        type="string",
    ),
    AttributeInfo(
        name="severity",
        description="Ticket severity level. One of: low, medium, high, critical.",
        type="string",
    ),
    AttributeInfo(
        name="tenant_tier",
        description="Subscription tier. One of: Starter, Professional, Enterprise.",
        type="string",
    ),
    AttributeInfo(
        name="resolved",
        description="Whether the ticket is resolved.",
        type="boolean",
    ),
    AttributeInfo(
        name="duration_minutes",
        description="Duration of the issue in minutes.",
        type="integer",
    ),
]

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm_client,
    vectorstore=tickets_store,
    document_contents="Datanous.ai support ticket description",
    metadata_field_info=metadata_field_info,
    enable_limit=True,
)

# Natural language filter queries
test_queries = [
    "unresolved critical Datanous Guard tickets",
    "high severity Datanous Search issues for Professional clients",
    "any Datanous Embed tickets that lasted more than 15 minutes",
]

for q in test_queries:
    print(f"Query: {q}")
    results = self_query_retriever.invoke(q)

    if not results:
        print("  No results found.")

    for r in results:
        m = r.metadata
        print(
            f"  [{m.get('product')} | {m.get('severity')} | "
            f"tier={m.get('tenant_tier')} | resolved={m.get('resolved')} | "
            f"duration={m.get('duration_minutes')}] {r.page_content}"
        )
    print()

Query: unresolved critical Datanous Guard tickets
  [Datanous Guard | critical | tier=Enterprise | resolved=False | duration=47] Critical unresolved incident: Datanous Guard blocked legitimate tenant requests after a policy update.
  [Datanous Insight | critical | tier=Professional | resolved=False | duration=63] Datanous Insight Professional tenant reported cross-tenant rows because tenant_id middleware was disabled.

Query: high severity Datanous Search issues for Professional clients
  [Datanous Search | high | tier=Professional | resolved=True | duration=18] High severity Datanous Search issue: metadata filter syntax caused empty result sets for a Professional tenant.

Query: any Datanous Embed tickets that lasted more than 15 minutes
  [Datanous Embed | medium | tier=Starter | resolved=True | duration=22] Datanous Embed returned 429 errors because the client exceeded the maximum batch size.



## 4.6 Routing patterns - best practices

| Pattern | Πότε ταιριάζει | Πλεονέκτημα |
|---|---|---|
| **LLM-based logical routing** | Πολλές ξεχωριστές πηγές | Ευέλικτο, μπορεί να εξηγήσει την επιλογή |
| **Embedding-based semantic routing** | Επιλογή prompt/persona | Γρήγορο, χωρίς LLM call |
| **Rule-based / regex** | Σαφή keywords (πχ «PRICING» στην ερώτηση) | Deterministic, μηδαμινό cost |
| **Hybrid (rules → LLM fallback)** | Production-ready | Βέλτιστος συνδυασμός cost/accuracy |

**Pro tip:** Σε production συνδύασε rules-first για 80% των cases (zero LLM cost) και LLM router για το υπόλοιπο 20%.

## 4.7 Άσκηση

**Άσκηση:** Φτιάξε ένα `HybridRouter` class που:

1. Πρώτα ψάχνει για keyword patterns (πχ «cost», «τιμή», «$» → pricing)
2. Αν δεν match-άρει, καλεί τον LLM router
3. Επιστρέφει το όνομα του datasource

Δοκιμάσε τον σε διάφορες ερωτήσεις και μέτρα πόσες πιάστηκαν από τους κανόνες vs LLM.

In [14]:
# Exercise: Build a HybridRouter class for Datanous.ai that:
# 1. First tries keyword-based routing using regex patterns:
#    - Pricing keywords (price, cost, USD, plan, billing) -> "pricing_docs"
#    - Error keywords (error, 429, 503, failed, timeout) -> "support_docs"
#    - Otherwise -> fallback to LLM router
# 2. If no keyword matches, use the router LLM (already defined above)
# 3. Track how many requests were routed by rules vs LLM
#
# Test with:
#   hr = HybridRouter(router)
#   hr.route("How much does the Enterprise plan cost?")     # should hit pricing rule
#   hr.route("Getting 429 errors from Datanous Embed")      # should hit support rule
#   hr.route("What is the architecture of Datanous Guard?") # should fall through to LLM

# Your solution here:


In [15]:
import re

class HybridRouter:
    """Routes Datanous.ai support questions using keyword rules first, LLM as fallback."""

    RULES = [
        (re.compile(r"\b(price|cost|USD|plan|billing|tier|discount|subscription)\b", re.I),
         "pricing_docs"),
        (re.compile(r"\b(error|exception|failed|timeout|429|503|404|not working|issue|bug)\b", re.I),
         "support_docs"),
        (re.compile(r"\b(api|sdk|configure|parameter|architecture|integration|endpoint)\b", re.I),
         "product_docs"),
    ]

    def __init__(self, llm_router):
        self.llm_router = llm_router
        self.stats = {"rules": 0, "llm": 0}

    def route(self, question: str) -> str:
        for pattern, target in self.RULES:
            if pattern.search(question):
                self.stats["rules"] += 1
                print(f"  [rule] -> {target}")
                return target
        self.stats["llm"] += 1
        decision = self.llm_router.invoke({"question": question})
        print(f"  [llm]  -> {decision.datasource} ({decision.reasoning[:60]})")
        return decision.datasource

hybrid = HybridRouter(router)

test_questions = [
    "How much does the Datanous Insight Enterprise plan cost?",
    "Getting 429 errors from the Datanous Embed API",
    "What is the architecture of Datanous Guard middleware?",
    "Does Datanous.ai offer on-premises deployment?",
]
for q in test_questions:
    print(f"Q: {q}")
    hybrid.route(q)

print(f"\nRouting stats: {hybrid.stats}")


Q: How much does the Datanous Insight Enterprise plan cost?
  [rule] -> pricing_docs
Q: Getting 429 errors from the Datanous Embed API
  [rule] -> support_docs
Q: What is the architecture of Datanous Guard middleware?
  [rule] -> product_docs
Q: Does Datanous.ai offer on-premises deployment?
  [llm]  -> product_docs (The question pertains to the deployment options available fo)

Routing stats: {'rules': 3, 'llm': 1}


## 📋 Συμπεράσματα

| # | Έννοια | Συνοπτικά |
|---|---|---|
| 1 | Πρόβλημα | Πολλές πηγές γνώσης / πολλοί τύποι ερωτήσεων |
| 2 | Logical routing | LLM επιστρέφει ποιο datasource, με `with_structured_output()` |
| 3 | Pydantic Literal | Περιορίζει τις επιτρεπτές απαντήσεις σε enum |
| 4 | Semantic routing | Embedding similarity με template prompts → επιλογή persona |
| 5 | Query construction | NL → structured Pydantic query με metadata filters |
| 6 | Vector + filter | Συνδυασμός semantic search + structured filtering |
| 7 | SelfQueryRetriever | Έτοιμο abstraction του LangChain |
| 8 | Hybrid router | Rules-first + LLM fallback = cost-optimal |
| 9 | Επόμενο βήμα | Notebook 05 — Advanced Indexing |